# ComfyUI セットアップ（PixelArt / Flux.1 Dev / 24GB専用）

## 構成

- **コンテナ内（毎回消える）**: 全てのデータ（ComfyUI本体・モデル・カスタムノードなど）
- **バックアップzip経由でローカル保存**: ワークフロー・設定・トークンなど

## 運用フロー



## Cell 1: 基本設定・バックアップzip展開・トークン読み込み

In [ ]:
# ========================================
# ★ ここでGPU TIERを選択 ★
# 24GB専用（RTX 4090等）→ Flux.1 Dev FP8 + PixelArt LoRA x2
# ========================================
GPU_TIER = "24GB"

import os, shutil, subprocess, zipfile, glob
from pathlib import Path

WORK_DIR = "/workspace/runpod-slim"
os.makedirs(WORK_DIR, exist_ok=True)

if GPU_TIER not in ["16GB", "24GB", "32GB", "48GB"]:
    raise SystemExit(f"❌ GPU_TIER は 16GB/24GB/32GB/48GB のいずれかにしてください（現在: {GPU_TIER}）")

# ===== バックアップzip自動展開 =====
zip_candidates = sorted(glob.glob(f"{WORK_DIR}/comfyui_backup_*.zip"))
if zip_candidates:
    zip_path = zip_candidates[-1]  # 最新のzipを使う
    print(f"📦 バックアップzip検出: {os.path.basename(zip_path)}")
    print(f"   展開中...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(WORK_DIR)
    print(f"   ✅ 展開完了")
    # 展開済みzipは別ファイル名にリネーム（次回再展開を防ぐ）
    extracted_marker = zip_path + ".extracted"
    os.rename(zip_path, extracted_marker)
    print(f"   ✅ {os.path.basename(extracted_marker)} にリネーム済み")
else:
    print("ℹ️  バックアップzipなし（初回起動扱い）")

# wildcards不使用（PixelArt専用構成）

# ===== comfyui_pixelart.html を WORK_DIR へコピー =====
HTML_SRC = f"{WORK_DIR}/comfyui_pixelart.html"
if os.path.exists(HTML_SRC):
    print(f"✅ comfyui_pixelart.html 確認済み")
else:
    print("⚠️  comfyui_pixelart.html が見つかりません（バックアップに含まれているか確認してください）")

# ===== .env 管理（HF_TOKEN, CIVITAI_TOKEN, DISCORD_WEBHOOK_URL） =====
ENV_FILE = f"{WORK_DIR}/.env"

def load_env():
    env = {}
    if os.path.exists(ENV_FILE):
        with open(ENV_FILE) as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                env[k.strip()] = v.strip().strip('"').strip("'")
    return env

def save_env(env):
    with open(ENV_FILE, "w") as f:
        for k, v in env.items():
            f.write(f'{k}="{v}"\n')
    os.chmod(ENV_FILE, 0o600)
    print(f"✅ トークンを {ENV_FILE} に保存しました")

env = load_env()
HF_TOKEN          = env.get("HF_TOKEN", "")
CIVITAI_TOKEN     = env.get("CIVITAI_TOKEN", "")
DISCORD_WEBHOOK_URL = env.get("DISCORD_WEBHOOK_URL", "")
JUPYTER_PASSWORD  = env.get("JUPYTER_PASSWORD", "")

# CIVITAI_TOKENが未設定の場合、上の "YOUR_CIVITAI_TOKEN_HERE" を実際のキーに書き換えてください
# .envファイルに保存済みの場合はそちらが優先されます

# ===== トークン確認 =====
missing = []
if not HF_TOKEN:
    missing.append("HF_TOKEN")
if not CIVITAI_TOKEN:
    missing.append("CIVITAI_TOKEN")
if not DISCORD_WEBHOOK_URL:
    missing.append("DISCORD_WEBHOOK_URL")
if not JUPYTER_PASSWORD:
    missing.append("JUPYTER_PASSWORD")

if missing:
    print(f"\n⚠️  未設定: {', '.join(missing)}")
    print("\n   下記コードを新しいセルに貼って実行してください：\n")
    print("   env = load_env()")
    print('   env["HF_TOKEN"] = "hf_xxxxxxxxxxxxxxxx"')
    print('   env["CIVITAI_TOKEN"] = "xxxxxxxx"  # 任意')
    print('   env["DISCORD_WEBHOOK_URL"] = "https://discord.com/api/webhooks/..."')
    print("   save_env(env)")
else:
    print(f"\n✅ HF_TOKEN 読み込み済み（{HF_TOKEN[:7]}...）")
    print(f"✅ DISCORD_WEBHOOK_URL 読み込み済み")

# ===== パス設定 =====
COMFY_DIR     = f"{WORK_DIR}/ComfyUI"
MODEL_DIR     = "/comfyui_models"
WORKFLOWS_DIR = f"{COMFY_DIR}/user/default/workflows"

print(f"\n✅ GPU TIER  : {GPU_TIER}")
print(f"📂 WORK_DIR  : {WORK_DIR}（コンテナ内、毎回消える）")
print(f"📂 COMFY_DIR : {COMFY_DIR}")
print(f"📂 MODEL_DIR : {MODEL_DIR}（コンテナ内、毎回消える）")
print(f"💾 ディスク空き: {shutil.disk_usage(WORK_DIR).free / 1e9:.1f} GB")
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True
    ).stdout.strip()
    print(f"🎮 GPU: {gpu}")
except Exception:
    print("⚠️  nvidia-smi なし")

## Cell 1.5: Jupyterパスワード自動設定


In [ ]:
# ===== Jupyter パスワード自動設定 =====
import json as _json, os

_jupyter_pw = env.get("JUPYTER_PASSWORD", "")
if _jupyter_pw:
    _config_path = "/root/.jupyter/jupyter_server_config.json"
    from jupyter_server.auth.security import passwd
    _hashed = passwd(_jupyter_pw)
    _config = {}
    if os.path.exists(_config_path):
        with open(_config_path) as f:
            _config = _json.load(f)
    _config.setdefault("ServerApp", {})["password"] = _hashed
    os.makedirs(os.path.dirname(_config_path), exist_ok=True)
    with open(_config_path, "w") as f:
        _json.dump(_config, f)
    print("✅ Jupyterパスワード設定完了")
else:
    print("⚠️ JUPYTER_PASSWORD未設定（.envに追加してください）")


## Cell 2: ComfyUI本体配置（毎回必要）

In [ ]:
# ComfyUI本体のtrackingブランチ設定（cm-cli update all に必要）
import subprocess, os

COMFY_DIR_GIT = COMFY_DIR  # Cell 1で定義済み

result = subprocess.run(
    ["git", "rev-parse", "--abbrev-ref", "--symbolic-full-name", "@{u}"],
    cwd=COMFY_DIR_GIT, capture_output=True, text=True
)

if result.returncode != 0:
    # tracking未設定 → fetch→設定
    subprocess.run(["git", "fetch", "origin"], cwd=COMFY_DIR_GIT, capture_output=True)
    subprocess.run(["git", "remote", "set-head", "origin", "-a"], cwd=COMFY_DIR_GIT, capture_output=True)
    remote_head = subprocess.run(
        ["git", "symbolic-ref", "refs/remotes/origin/HEAD"],
        cwd=COMFY_DIR_GIT, capture_output=True, text=True
    ).stdout.strip()
    remote_branch = remote_head.split("/")[-1] if remote_head else "master"
    local_branch = subprocess.run(
        ["git", "rev-parse", "--abbrev-ref", "HEAD"],
        cwd=COMFY_DIR_GIT, capture_output=True, text=True
    ).stdout.strip()
    r = subprocess.run(
        ["git", "branch", "--set-upstream-to", f"origin/{remote_branch}", local_branch],
        cwd=COMFY_DIR_GIT, capture_output=True, text=True
    )
    if r.returncode == 0:
        print(f"✅ ComfyUI本体 tracking設定完了 (origin/{remote_branch})")
    else:
        print(f"⚠️ ComfyUI本体 tracking設定失敗: {r.stderr.strip()[:100]}")
else:
    print(f"✅ ComfyUI本体 tracking設定済み ({result.stdout.strip()})")


## Cell 3: extra_model_paths.yaml 生成

In [ ]:
# extra_model_paths.yaml 生成
import os
from pathlib import Path

os.makedirs(WORKFLOWS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

yaml_path = Path(f"{COMFY_DIR}/extra_model_paths.yaml")
yaml_content = f"""comfyui:
    base_path: {MODEL_DIR}/
    is_default: true
    checkpoints: checkpoints/
    clip: text_encoders/
    text_encoders: text_encoders/
    clip_vision: clip_vision/
    configs: configs/
    controlnet: controlnet/
    diffusion_models: |
        diffusion_models/
        unet/
    embeddings: embeddings/
    loras: loras/
    upscale_models: upscale_models/
    vae: vae/
    ipadapter: ipadapter/
    ultralytics: ultralytics/
    ultralytics_bbox: ultralytics/bbox/
    ultralytics_segm: ultralytics/segm/
"""
with open(yaml_path, "w") as f:
    f.write(yaml_content)
print(f"✅ Cell 3 完了 — extra_model_paths.yaml 生成 ({MODEL_DIR}/)")
# wildcards フォルダ作成 + world_setting.txt 配置
import shutil as _shutil
WILDCARDS_DIR = f"{COMFY_DIR}/wildcards"
os.makedirs(WILDCARDS_DIR, exist_ok=True)
_ws_src = f"{WORK_DIR}/world_setting.txt"
_ws_dst = f"{WILDCARDS_DIR}/world_setting.txt"
if os.path.exists(_ws_src):
    _shutil.copy2(_ws_src, _ws_dst)
    print(f"✅ world_setting.txt → {WILDCARDS_DIR}/")
else:
    print("⚠️ world_setting.txt が見つかりません（WORK_DIRに配置してください）")
# wildcardファイルを一括コピー
_wc_src_dir = f"{WORK_DIR}/wildcards"
if os.path.exists(_wc_src_dir):
    for _wc_file in os.listdir(_wc_src_dir):
        if _wc_file.endswith(".txt"):
            _shutil.copy2(f"{_wc_src_dir}/{_wc_file}", f"{WILDCARDS_DIR}/{_wc_file}")
    print(f"✅ wildcards/*.txt → {WILDCARDS_DIR}/")
else:
    print("ℹ️ wildcards/ フォルダなし（スキップ）")

## Cell 3.5: ワークフロー内のモデル名をGPU_TIERに応じて自動置換

In [ ]:
# ワークフロー内のモデル名をGPU_TIERに応じて自動置換
import json, os

TIER_MODELS = {
    "24GB": {
        "__DIFFUSION_MODEL__": "flux1-dev-fp8.safetensors",
        "__CLIP1_MODEL__":     "clip_l.safetensors",
        "__CLIP2_MODEL__":     "t5xxl_fp8_e4m3fn.safetensors",
        "__CHECKPOINT_MODEL__": "",
        "__OLLAMA_MODEL__":    "jaahas/qwen3.5-uncensored:9b",
        "target_workflow":     "pixelart_24GB_workflow_v1.json",
    },
}

tier_config  = TIER_MODELS[GPU_TIER]
target_fname = tier_config["target_workflow"]
replacements = {k: v for k, v in tier_config.items() if k != "target_workflow"}

if not os.path.exists(WORKFLOWS_DIR):
    print(f"⚠️  workflowsディレクトリなし: {WORKFLOWS_DIR}")
else:
    target_path = f"{WORKFLOWS_DIR}/{target_fname}"
    if not os.path.exists(target_path):
        print(f"⚠️  ワークフローファイルなし: {target_fname}")
    else:
        with open(target_path, encoding="utf-8") as f:
            content = f.read()
        for placeholder, real_name in replacements.items():
            content = content.replace(placeholder, real_name)
        with open(target_path, "w", encoding="utf-8") as f:
            f.write(content)

if os.path.exists(WORKFLOWS_DIR):
    for fname in os.listdir(WORKFLOWS_DIR):
        fpath = os.path.join(WORKFLOWS_DIR, fname)
        if fname.endswith(".bak") or (not fname.endswith(".json") and os.path.isfile(fpath)):
            os.remove(fpath)

print(f"✅ Cell 3.5 完了 — {target_fname} 置換・クリーンアップ完了")

## Cell 3.7: NGカスタムノードをリストから除外

In [ ]:
# NGノードをcustom_nodes.txtから除外
NG_NODES = [
    "FantasyTalking",
]
import os as _os
_cn_txt = f"{WORK_DIR}/custom_nodes.txt"
if _os.path.exists(_cn_txt):
    with open(_cn_txt) as f:
        _lines = f.readlines()
    _lines = [l for l in _lines if not any(ng in l for ng in NG_NODES)]
    with open(_cn_txt, 'w') as f:
        f.writelines(_lines)
    print(f'✅ Cell 3.7 完了 — NGノード除外済み ({len(NG_NODES)}件)')
else:
    print('ℹ️ custom_nodes.txt なし（初回起動）')

## Cell 4: カスタムノード自動クローン

In [ ]:
# カスタムノード自動クローン
import os, shutil, subprocess

CUSTOM_NODES_LIST = f"{WORK_DIR}/custom_nodes.txt"

if not os.path.exists(CUSTOM_NODES_LIST):
    default_nodes = [
        "# === コアノード ===",
        "https://github.com/ltdrdata/ComfyUI-Manager",
        "https://github.com/city96/ComfyUI-GGUF",
        "https://github.com/stavsap/comfyui-ollama",
        "",
        "# === UI改善 ===",
        "https://github.com/pythongosssss/ComfyUI-Custom-Scripts",
        "",
        "# === Face Detailer ===",
        "https://github.com/ltdrdata/ComfyUI-Impact-Pack",
        "https://github.com/ltdrdata/ComfyUI-Impact-Subpack",
        "",
        "# === アップスケール ===",
        "https://github.com/ssitu/ComfyUI_UltimateSDUpscale",
        "",
        "# === 高度なサンプリング ===",
        "https://github.com/ltdrdata/ComfyUI-Inspire-Pack",
        "",
        "# === インペイント ===",
        "https://github.com/Acly/comfyui-inpaint-nodes",
        "",
        "# === IP-Adapter（参照画像）===",
        "https://github.com/cubiq/ComfyUI_IPAdapter_plus",
        "",
        "# === ControlNet前処理 ===",
        "https://github.com/Fannovel16/comfyui_controlnet_aux",
        "",
        "# === 画像→テキスト（自然文）===",
        "https://github.com/kijai/ComfyUI-Florence2",
        "",
        "# === 画像→テキスト（タグ式）===",
        "https://github.com/pythongosssss/ComfyUI-WD14-Tagger",
        "",
        "# === ワイルドカード ===",
        "https://github.com/a-und-b/ComfyUI_AB_Wildcard.git",
        "\n",
        "# === rgthree（Power Lora Loader）===\n",
        "https://github.com/rgthree/rgthree-comfy",
    ]
    with open(CUSTOM_NODES_LIST, "w") as f:
        f.write("\n".join(default_nodes) + "\n")

custom_nodes_dir = f"{COMFY_DIR}/custom_nodes"
os.makedirs(custom_nodes_dir, exist_ok=True)

with open(CUSTOM_NODES_LIST) as f:
    urls = [line.strip() for line in f if line.strip() and not line.startswith("#")]

ok, ng = [], []
for url in urls:
    name = url.rstrip("/").split("/")[-1].replace(".git", "")
    target = os.path.join(custom_nodes_dir, name)
    if os.path.exists(target):
        # トラッキングブランチが未設定の場合は設定する
        try:
            result = subprocess.run(
                ["git", "rev-parse", "--abbrev-ref", "--symbolic-full-name", "@{u}"],
                cwd=target, capture_output=True, text=True
            )
            if result.returncode != 0:
                # トラッキングブランチ未設定 → 設定する
                subprocess.run(["git", "remote", "set-head", "origin", "-a"],
                               cwd=target, capture_output=True)
                branch = subprocess.run(
                    ["git", "rev-parse", "--abbrev-ref", "HEAD"],
                    cwd=target, capture_output=True, text=True
                ).stdout.strip()
                subprocess.run(
                    ["git", "branch", "--set-upstream-to", f"origin/{branch}", branch],
                    cwd=target, capture_output=True
                )
                print(f"✅ skip  {name} (tracking設定済み)")
            else:
                print(f"✅ skip  {name}")
        except Exception as e:
            print(f"✅ skip  {name} (tracking設定失敗: {e})")
        ok.append(name)
        continue
    try:
        subprocess.run(["git", "clone", url, target],
                       check=True, capture_output=True)
        # clone直後にトラッキングブランチを明示設定
        subprocess.run(["git", "remote", "set-head", "origin", "-a"],
                       cwd=target, capture_output=True)
        branch = subprocess.run(
            ["git", "rev-parse", "--abbrev-ref", "HEAD"],
            cwd=target, capture_output=True, text=True
        ).stdout.strip()
        subprocess.run(
            ["git", "branch", "--set-upstream-to", f"origin/{branch}", branch],
            cwd=target, capture_output=True
        )
        print(f"✅ clone {name}")
        ok.append(name)
    except subprocess.CalledProcessError:
        print(f"❌ fail  {name}")
        ng.append(name)

print(f"\n✅ Cell 4 完了 — {len(ok)} 件成功 / {len(ng)} 件失敗")


## Cell 5: pip依存関係の再インストール（毎回必須）

In [ ]:
# pip依存関係の再インストール（毎回必須）
import subprocess, os
from pathlib import Path

def pip_install(req_path, label):
    r = subprocess.run(["pip", "install", "-q", "-r", str(req_path)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f"❌ {label}")
        print(r.stderr[-500:] if r.stderr else r.stdout[-500:])
    else:
        print(f"✅ {label}")

req = f"{COMFY_DIR}/requirements.txt"
if os.path.exists(req):
    pip_install(req, "ComfyUI本体")

custom_nodes_path = Path(f"{COMFY_DIR}/custom_nodes")
if custom_nodes_path.exists():
    for d in sorted(custom_nodes_path.iterdir()):
        if d.is_dir():
            req = d / "requirements.txt"
            if req.exists():
                pip_install(req, d.name)

# 追加パッケージ（requirements.txtに含まれないもの）
for pkg in ["ultralytics", "rembg[gpu]"]:
    r = subprocess.run(["pip", "install", "-q", pkg, "--break-system-packages"], capture_output=True, text=True)
    print(f"✅ {pkg}" if r.returncode == 0 else f"❌ {pkg}: {r.stderr[-200:]}")

print("\n✅ Cell 5 完了")

## Cell 5.5: カスタムノード Update All（cm-cli）

In [ ]:
# カスタムノードをComfyUI Manager経由で一括アップデート
import subprocess, os

CM_CLI = f"{COMFY_DIR}/custom_nodes/ComfyUI-Manager/cm-cli.py"

if os.path.exists(CM_CLI):
    print("🔄 cm-cli update all 実行中...")
    result = subprocess.run(
        ["python3", CM_CLI, "update", "all"],
        cwd=COMFY_DIR,
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("✅ Cell 5.5 完了 — 全カスタムノード更新済み")
    else:
        print(f"⚠️ cm-cli エラー（続行します）:\n{result.stderr[-300:]}")
else:
    print("⚠️ cm-cli.py が見つかりません（スキップ）")


## Cell 6.5: Ollama 自動セットアップ（インストール・サーバー起動）

In [ ]:
# ===== Ollama 自動セットアップ =====
import subprocess, time, requests, os, shutil, sys

# Step 1: Ollamaインストール
if not shutil.which("ollama"):
    print("[1/3] Ollamaインストール中...")
    try:
        try:
            get_ipython().system('apt-get update -qq')
            get_ipython().system('apt-get install -y -qq zstd')
            get_ipython().system('curl -fsSL https://ollama.com/install.sh | sh')
        except NameError:
            subprocess.run("apt-get update -qq && apt-get install -y -qq zstd && curl -fsSL https://ollama.com/install.sh | sh",
                           shell=True, check=True, capture_output=True)
        print("[1/3] ✅ Ollamaインストール完了")
    except Exception as e:
        print(f"[1/3] ❌ インストール失敗: {e}")
        sys.exit(1)
else:
    print("[1/3] ✅ Ollama既にインストール済み")

# Step 2: モデル保存先設定
OLLAMA_MODELS_DIR = f"{WORK_DIR}/ollama_models"
os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)
os.environ["OLLAMA_MODELS"] = OLLAMA_MODELS_DIR
print(f"[2/3] ✅ OLLAMA_MODELS = {OLLAMA_MODELS_DIR}")

# Step 3: サーバー起動
already_running = False
try:
    r = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
    if r.status_code == 200:
        already_running = True
except Exception:
    pass

if already_running:
    print("[3/3] ✅ Ollamaサーバー起動済み")
else:
    print("[3/3] Ollamaサーバー起動中...")
    _env = {**os.environ, "OLLAMA_MODELS": OLLAMA_MODELS_DIR}
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=open(f"{WORK_DIR}/ollama.log", "w"),
        stderr=subprocess.STDOUT,
        env=_env
    )
    for i in range(20):
        try:
            r = requests.get("http://127.0.0.1:11434/api/tags", timeout=1)
            if r.status_code == 200:
                print(f"[3/3] ✅ 起動完了 ({i+1}秒)")
                break
        except Exception:
            pass
        time.sleep(1)
    else:
        print("[3/3] ⚠️ 起動タイムアウト（ollama.log を確認）")

print("\n✅ Cell 6.5 完了 — Ollamaインストール・サーバー起動")
print("ℹ️  モデルpullは download_ui.ipynb で実行してください")

## Cell 6.7: comfyui_mobile.html を Custom-Scripts/web/ に配置

In [ ]:
# comfyui_mobile.html を ComfyUI-Custom-Scripts/web/ に配置
# アクセスURL: https://{pod_id}-8188.proxy.runpod.net/extensions/ComfyUI-Custom-Scripts/comfyui_mobile.html
import os, re as _re67

pod_id_67 = os.environ.get("RUNPOD_POD_ID", "")
comfy_url_67 = f"https://{pod_id_67}-8188.proxy.runpod.net" if pod_id_67 else ""
jupyter_url_67 = f"https://{pod_id_67}-8888.proxy.runpod.net" if pod_id_67 else ""

_html_src = f"{WORK_DIR}/comfyui_mobile_full.html"
_html_src = _html_src if os.path.exists(_html_src) else f"{WORK_DIR}/comfyui_mobile.html"
_html_dst_dir = f"{COMFY_DIR}/custom_nodes/ComfyUI-Custom-Scripts/web"
_html_dst = f"{_html_dst_dir}/{os.path.basename(_html_src)}"

if os.path.exists(_html_src) and os.path.exists(_html_dst_dir):
    with open(_html_src, encoding="utf-8") as f:
        html = f.read()
    # 既存valueを除去してから新URLを埋め込む（重複防止）
    html = _re67.sub(r'(<input[^>]*id="serverUrl"[^>]*?)\s*value="[^"]*"([^>]*>)', r'\1\2', html)
    html = _re67.sub(
        r'(<input[^>]*id="serverUrl"[^>]*)(>)',
        lambda m: m.group(1) + f' value="{comfy_url_67}"' + m.group(2),
        html
    )
    html = _re67.sub(r'(<input[^>]*id="jupyterUrl"[^>]*?)\s*value="[^"]*"([^>]*>)', r'\1\2', html)
    html = _re67.sub(
        r'(<input[^>]*id="jupyterUrl"[^>]*)(>)',
        lambda m: m.group(1) + f' value="{jupyter_url_67}"' + m.group(2),
        html
    )
    os.makedirs(_html_dst_dir, exist_ok=True)
    with open(_html_dst, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✅ Cell 6.7 完了 — {os.path.basename(_html_src)} → Custom-Scripts/web/ に配置")
    # extensions/ にも同じファイルを配置（CORSアクセスURL用）
    _html_dst_dir2 = f"{COMFY_DIR}/custom_nodes/ComfyUI-Custom-Scripts/web/extensions/ComfyUI-Custom-Scripts"
    _html_dst2 = f"{_html_dst_dir2}/{os.path.basename(_html_src)}"
    os.makedirs(_html_dst_dir2, exist_ok=True)
    with open(_html_dst2, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✅ Cell 6.7 追加 — {os.path.basename(_html_src)} → extensions/ComfyUI-Custom-Scripts/ に配置")
elif not os.path.exists(_html_src):
    print("⚠️ comfyui_mobile.html が見つかりません")
else:
    print("⚠️ ComfyUI-Custom-Scripts が未インストール（Cell 4を先に実行してください）")


## Cell 7: ComfyUI 起動 + Discord通知

In [ ]:
# ComfyUI 起動 + HTML URL書き込み + Discord通知
import time, requests as _req, shutil as _shutil, re as _re

LOG_FILE = f"{WORK_DIR}/comfyui.log"

# ポート競合対策: 既存プロセスを強制終了して待機
# ポート8188を使用しているプロセスを強制終了
import subprocess as _sp
# まずpkillで試みる
_sp.run(["pkill", "-9", "-f", "ComfyUI/main.py"], check=False)
# fuser でポート直接解放
_sp.run("fuser -k 8188/tcp 2>/dev/null || true", shell=True)
# ポートが解放されるまで待機（最大30秒）
import socket as _sock
for _i in range(30):
    try:
        s = _sock.socket()
        s.setsockopt(_sock.SOL_SOCKET, _sock.SO_REUSEADDR, 1)
        s.bind(("0.0.0.0", 8188))
        s.close()
        print(f"✅ ポート8188解放確認 ({_i+1}秒)")
        break
    except OSError:
        time.sleep(1)
else:
    print("⚠️ ポート8188が解放できません — 続行します")

TIER_FLAGS = {
    "16GB": ["--lowvram"],
    "24GB": ["--reserve-vram", "1"],
    "32GB": [],
    "48GB": [],
}
extra_flags = TIER_FLAGS.get(GPU_TIER, [])
log_fp = open(LOG_FILE, "w")
proc = subprocess.Popen(
    ["python3", "main.py", "--listen", "0.0.0.0", "--port", "8188",
     "--extra-model-paths-config", f"{COMFY_DIR}/extra_model_paths.yaml",
     "--enable-cors-header"] + extra_flags,
    cwd=COMFY_DIR,
    stdout=log_fp,
    stderr=subprocess.STDOUT,
)

print(f"🚀 ComfyUI 起動中 (PID={proc.pid}) ... 15秒待ちます")
time.sleep(15)

# エラー行のみ表示
with open(LOG_FILE) as f:
    lines = f.readlines()
err_lines = [l.rstrip() for l in lines if any(kw in l for kw in ["ERROR", "CRITICAL", "Traceback", "Exception"])]
if err_lines:
    print("⚠️ ログにエラーあり:")
    for l in err_lines:
        print(f"  {l}")
else:
    print("✅ 起動ログ正常（エラーなし）")

# URL生成
pod_id = os.environ.get("RUNPOD_POD_ID", "")
comfy_url = f"https://{pod_id}-8188.proxy.runpod.net" if pod_id else "URLが取得できませんでした"
jupyter_url = f"https://{pod_id}-8888.proxy.runpod.net" if pod_id else ""
print(f"\n✅ ComfyUI URL: {comfy_url}")

# Discord通知
if DISCORD_WEBHOOK_URL:
    try:
        _req.post(DISCORD_WEBHOOK_URL, json={"content": (
            f"✅ ComfyUI起動完了\n"
            f"🔗 ComfyUI: {comfy_url}\n"
            f"📱 モバイル: {comfy_url}/extensions/ComfyUI-Custom-Scripts/comfyui_mobile.html\n"
            f"🖥️ Jupyter: {jupyter_url}\n"
            f"⚙️ TIER: {GPU_TIER}"
        )}, timeout=10)
        print("📱 Discord通知送信済み")
    except Exception as e:
        print(f"⚠️ Discord通知失敗: {e}")
else:
    print("⚠️ DISCORD_WEBHOOK_URL 未設定")

## Cell 8: 手動アップデート（必要な時だけ）

In [ ]:
# ===== アップデート（必要な時だけ）=====
# ⚠️ このセルは手動実行専用です。「全て実行」では自動スキップされます。
raise SystemExit("手動実行専用セルです。実行したい場合はこの行を削除してから実行してください。")

import time
import requests as _req

def git_update(repo_dir, name):
    print(f"  {name}")
    is_shallow = subprocess.run(
        ["git", "-C", repo_dir, "rev-parse", "--is-shallow-repository"],
        capture_output=True, text=True
    ).stdout.strip() == "true"
    if is_shallow:
        subprocess.run(["git", "-C", repo_dir, "fetch", "--depth=1"], check=False)
        branch = subprocess.run(
            ["git", "-C", repo_dir, "rev-parse", "--abbrev-ref", "HEAD"],
            capture_output=True, text=True
        ).stdout.strip() or "main"
        subprocess.run(["git", "-C", repo_dir, "reset", "--hard", f"origin/{branch}"], check=False)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull"], check=False)

# Step 1: アップデート
print("🔄 ComfyUI本体をアップデート")
if (Path(COMFY_DIR) / ".git").exists():
    git_update(COMFY_DIR, "ComfyUI")

print("\n🔄 カスタムノードをアップデート")
for d in sorted(Path(f"{COMFY_DIR}/custom_nodes").iterdir()):
    if d.is_dir() and (d / ".git").exists():
        git_update(str(d), d.name)

# Step 2: pip再インストール
print("\n📦 pip依存関係を再インストール...")
req = f"{COMFY_DIR}/requirements.txt"
if os.path.exists(req):
    subprocess.run(["pip", "install", "-q", "-r", req], check=False)
    print("  ✅ ComfyUI本体")
custom_nodes_path = Path(f"{COMFY_DIR}/custom_nodes")
if custom_nodes_path.exists():
    for d in sorted(custom_nodes_path.iterdir()):
        if d.is_dir():
            req = d / "requirements.txt"
            if req.exists():
                subprocess.run(["pip", "install", "-q", "-r", str(req)], check=False)
                print(f"  ✅ {d.name}")

# Step 3: ComfyUI再起動
print("\n🔄 ComfyUI再起動...")
subprocess.run(["pkill", "-f", "ComfyUI/main.py"], check=False)
time.sleep(2)

LOG_FILE = f"{WORK_DIR}/comfyui.log"
log_fp = open(LOG_FILE, "w")
TIER_FLAGS = {
    "16GB": ["--lowvram"],
    "24GB": ["--reserve-vram", "1"],
    "32GB": [],
    "48GB": [],
}
extra_flags = TIER_FLAGS.get(GPU_TIER, [])
proc = subprocess.Popen(
    ["python3", "main.py", "--listen", "0.0.0.0", "--port", "8188",
     "--extra-model-paths-config", f"{COMFY_DIR}/extra_model_paths.yaml",
     "--enable-cors-header"] + extra_flags,
    cwd=COMFY_DIR,
    stdout=log_fp,
    stderr=subprocess.STDOUT,
)
print(f"⏳ ComfyUI 起動中 (PID={proc.pid})")
print("  15秒待機中...")
time.sleep(15)

with open(LOG_FILE) as f:
    lines = f.readlines()
err_lines = [l.rstrip() for l in lines if any(kw in l for kw in ["ERROR", "CRITICAL", "Traceback", "Exception"])]
if err_lines:
    print("⚠️ ログにエラーあり:")
    for l in err_lines:
        print(f"  {l}")
else:
    print("✅ 起動ログ正常（エラーなし）")

# Step 4: Discord通知
pod_id = os.environ.get("RUNPOD_POD_ID", "")
comfy_url = f"https://{pod_id}-8188.proxy.runpod.net" if pod_id else "URLが取得できませんでした"
print(f"\n✅ ComfyUI 起動完了: {comfy_url}")
if DISCORD_WEBHOOK_URL:
    try:
        _req.post(DISCORD_WEBHOOK_URL, json={"content": (
            f"✅ ComfyUI再起動完了（アップデート後）\n"
            f"🔗 {comfy_url}\n"
            f"⚙️ GPU: {GPU_TIER}"
        )}, timeout=10)
        print("✅ Discord通知 送信済み")
    except Exception as e:
        print(f"⚠️ Discord通知 失敗: {e}")

## Cell 9: 生成完了Discord通知（バックグラウンド監視）

In [ ]:
# ===== Cell 9: 生成完了Discord通知（バックグラウンド監視）=====
import threading, requests as _req, json as _json, time as _time
import os as _os, re as _re

_COMFY_URL = f"http://127.0.0.1:8188"
_DISCORD_URL = DISCORD_WEBHOOK_URL
_OUTPUT_DIR = f"{COMFY_DIR}/output"
_notified = set()
_stop_flag = [False]

def _get_lora_from_prompt(prompt_data):
    """プロンプトデータからLoRA名を抽出"""
    loras = []
    try:
        for node in prompt_data.values():
            if isinstance(node, dict):
                cls = node.get("class_type", "")
                if "LoraLoader" in cls:
                    lora = node.get("inputs", {}).get("lora_name", "")
                    if lora:
                        loras.append(_os.path.splitext(_os.path.basename(lora))[0])
    except Exception:
        pass
    return ", ".join(loras) if loras else "-"

def _get_positive_prompt(prompt_data):
    """CLIPTextEncodeのPositiveプロンプトを抽出"""
    try:
        for node in prompt_data.values():
            if isinstance(node, dict):
                if node.get("class_type") == "CLIPTextEncode":
                    text = node.get("inputs", {}).get("text", "")
                    if text and len(text) > 10:
                        return text[:80] + ("..." if len(text) > 80 else "")
    except Exception:
        pass
    return "-"

def _monitor():
    print("👀 キュー監視開始")
    while not _stop_flag[0]:
        try:
            r = _req.get(f"{_COMFY_URL}/history", timeout=5)
            if r.status_code != 200:
                _time.sleep(3)
                continue
            history = r.json()
            for pid, item in history.items():
                if pid in _notified:
                    continue
                outputs = item.get("outputs", {})
                images = []
                for node_out in outputs.values():
                    for img in node_out.get("images", []):
                        if img.get("type") == "output":
                            images.append(img["filename"])
                if not images:
                    continue
                fname = images[-1]
                # 連番と日付抽出
                num_match = _re.search(r'_(\d+)_', fname)
                num = num_match.group(1) if num_match else "?"
                from datetime import datetime, timezone, timedelta, timezone, timedelta
                date_str = datetime.now(timezone(timedelta(hours=9))).strftime("%m-%d")
                # 所要時間
                elapsed = "-"
                timing = item.get("status", {}).get("execution_start_timestamp")
                timing_end = item.get("status", {}).get("execution_end_timestamp")
                if timing and timing_end:
                    elapsed = f"{int(timing_end - timing)}秒"
                # プロンプト・LoRA
                prompt_data = item.get("prompt", [{}])[-1] if item.get("prompt") else {}
                lora = _get_lora_from_prompt(prompt_data)
                prompt_text = _get_positive_prompt(prompt_data)
                # Discord送信
                if _DISCORD_URL:
                    msg = (
                        f"✅ #{num}  {date_str}\n"
                        f"🎨 {lora}\n"
                        f"📝 {prompt_text}\n"
                        f"⏱️ {elapsed}"
                    )
                    try:
                        _req.post(_DISCORD_URL, json={"content": msg}, timeout=5)
                    except Exception:
                        pass
                _notified.add(pid)
        except Exception:
            pass
        _time.sleep(3)

_stop_flag[0] = False
_t = threading.Thread(target=_monitor, daemon=True)
_t.start()
print("✅ Cell 9 完了 — 生成完了をDiscordに通知します")
print("   停止するには: _stop_flag[0] = True")